# 02 — LDA em Folha de São Paulo (PT-BR)

## O que esse notebook faz

Aplica **Latent Dirichlet Allocation (LDA)** ao corpus Folha de São Paulo.
Pipeline idêntico ao padrão dos demais notebooks LDA — corpus formal PT-BR.

**Corpus:** Folha de São Paulo — PT-BR, formal/jornalístico, 2018-2024.
**Entrada:** `data/input/folha/corpus_limpo.csv`


## 1. Setup e imports

Carregamos parâmetros do `configs/params.yaml` (entrada `corpora.folha`), o seed global e as funções dos pipelines reusáveis:

- `_helpers.lemmatize_corpus(docs, lang, params)` → tokens lematizados + Dictionary com filter_extremes
- `_helpers.grid_search_k / train_lda / extract_topics_keywords / compute_doc_distributions` → fit e extração
- `_helpers.name_all_topics` → rotulagem via Ollama gemma4:e4b
- `_helpers.*` → métricas (c_v, exclusivity, FREX, NPMI, etc.)

In [ ]:
# ── path fix: resolve _helpers.py e data/ de qualquer subdiretório ──
import sys as _sys, os as _os
from pathlib import Path as _P
_here = _P().resolve()
_nb_dir = _here if (_here / '_helpers.py').exists() else _here.parent
if str(_nb_dir) not in _sys.path:
    _sys.path.insert(0, str(_nb_dir))
_os.chdir(_nb_dir)          # '../data/...' paths ficam corretos
del _here, _nb_dir, _P, _sys, _os
# ─────────────────────────────────────────────────────────────────────
# Cell 1 — Imports
import sys, os, json, time, ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

from _helpers import load_params, get_corpus_config, get_seed, load_corpus
from _helpers import lemmatize_corpus
from _helpers import (
    grid_search_k, train_lda, extract_topics_keywords,
    compute_doc_distributions, qualitative_report,
)
from _helpers import name_all_topics
from _helpers import (
    compute_coherence_cv, compute_exclusivity, compute_diversity,
    compute_topic_diversity, export_results, export_topics_for_eval,
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100


In [ ]:
# Cell 2 — Carrega params do params.yaml
CORPUS_ID = "folha"
LANG = "pt"

params = load_params()
_, cfg = get_corpus_config(params, CORPUS_ID)
seed = get_seed(params)
TEXT_COL = cfg["text_column"]

print(f"corpus_id = {CORPUS_ID}")
print(f"text_column = {TEXT_COL}")
print(f"language = {LANG}")
print(f"seed = {seed}")
print(f"covariates (usadas no STM, não no LDA) = {cfg.get('covariates', [])}")

## 2. Corpus Folha de São Paulo

### Origem e schema
- **Fonte:** CSV em `data/raw/folha/`
- **Idioma:** PT-BR, formal/jornalístico, 2018-2024
- **Schema:** `post_id`, `message`, `date`, `category` (editoria)

| Aspecto | Reuters (EN) | Folha (PT-BR) |
|---|---|---|
| Idioma | EN | PT-BR |
| Registro | Financeiro | Jornalístico geral |
| spaCy | en_core_web_sm | pt_core_news_lg |


In [ ]:
# Cell 3 — Carrega corpus_limpo.csv
_corpus_dir = f"../data/input/{CORPUS_ID}/"
assert os.path.exists(input_path), (
    f"NÃO ENCONTRADO: {input_path}\n"
    f"Rode: python -m src.adapters.reuters em 01-preprocessing/"
)
df = load_corpus(_corpus_dir)
docs = df[TEXT_COL].astype(str).tolist()
post_ids = df["post_id"].astype(str).tolist()

print(f"Total docs: {len(docs):,}")
print(f"Colunas: {list(df.columns)}")
df.head(3)


In [ ]:
# Cell 4 — Estatisticas descritivas + distribuicao de tokens
df["n_tokens"] = df["message"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["n_tokens"], bins=60, edgecolor="white", color="steelblue")
axes[0].axvline(df["n_tokens"].median(), color="red", linestyle="--", label=f'mediana={df["n_tokens"].median():.0f}')
axes[0].axvline(df["n_tokens"].mean(), color="orange", linestyle="--", label=f'média={df["n_tokens"].mean():.0f}')
axes[0].set_xlabel("Tokens por documento"); axes[0].set_ylabel("# docs")
axes[0].set_title("Distribuição de tamanho dos documentos"); axes[0].legend()
axes[0].set_xlim(0, df["n_tokens"].quantile(0.99))

cat_counts = df["category"].value_counts().head(15)
axes[1].barh(cat_counts.index[::-1], cat_counts.values[::-1], color="steelblue")
axes[1].set_xlabel("# documentos"); axes[1].set_title("Top 15 categorias (primary label)")
plt.tight_layout(); plt.show()

print(f"\nDescritivas n_tokens: {df['n_tokens'].describe().to_dict()}")

## 3. Lematização e construção do vocabulário

### O que `src.lemmatize.lemmatize_corpus` faz

1. Carrega spaCy `en_core_web_sm` (cached em module-level)
2. Para cada doc: tokeniza, descarta stopwords (lista interna spaCy) + tokens não-alfa + tokens curtos (`len(lemma) <= 2`)
3. Constrói gensim `Dictionary` com `filter_extremes(no_below=5, no_above=0.5)`:
   - Remove palavras que aparecem em <5 docs (raras, ruído)
   - Remove palavras em >50% dos docs (genéricas, sem poder discriminante — ex.: "said")

### Por que lematizar (vs Porter stemming)

- Lematização preserva forma legível (`running` → `run`; STM via R usa Porter stemming agressivo: `running` → `run`, `companies` → `compani`)
- Mais interpretável para a banca da dissertação
- Custo: ~30s a ~3min em CPU dependendo do volume

In [ ]:
# Cell 5 — Lematiza corpus
print("Lematizando (single-process spaCy pt_core_news_lg)...")
t0 = time.time()
tokenized, dictionary = lemmatize_corpus(docs, LANG, params)
print(f"  done em {time.time()-t0:.0f}s")
print(f"  vocab final: {len(dictionary):,} palavras (após filter_extremes)")

# Estatísticas dos tokens lemmatizados
n_tokens_lemma = [len(t) for t in tokenized]
n_unique = [len(set(t)) for t in tokenized]

print(f"\n  avg tokens/doc após lemmatize: {np.mean(n_tokens_lemma):.1f}")
print(f"  avg unique tokens/doc: {np.mean(n_unique):.1f}")

In [ ]:
# Cell 6 — Top 30 palavras mais frequentes no vocabulário final
all_tokens = [t for doc in tokenized for t in doc]
freq = Counter(all_tokens)
top30 = freq.most_common(30)

fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*top30)
ax.barh(words[::-1], counts[::-1], color="steelblue")
ax.set_xlabel("Frequência total no corpus")
ax.set_title("Top 30 palavras (após lemmatize + filter_extremes)")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7 — Bag-of-Words representation
corpus_bow = [dictionary.doc2bow(d) for d in tokenized]
print(f"corpus_bow: {len(corpus_bow):,} docs")
print(f"avg unique tokens/doc no BoW: {np.mean([len(b) for b in corpus_bow]):.1f}")
print(f"\nExemplo doc[0] BoW (primeiros 10 pares (word_id, count)):")
for word_id, count in corpus_bow[0][:10]:
    print(f"  {dictionary[word_id]:20s}  count={count}")

## 4. Seleção de K (número de tópicos)

### Por que K importa

K é o **único hiperparâmetro estrutural** do LDA. Influencia a granularidade dos tópicos:
- K muito baixo → tópicos macro/genéricos (varios temas misturados em um)
- K muito alto → fragmentação (mesmo tema dividido em múltiplos clusters)

### Como escolher

Padrão na literatura: rodar LDA em vários K, calcular **c_v coherence** (Roder et al. 2015) — métrica que combina:
- co-ocorrência das top-N palavras numa janela deslizante
- score NPMI normalizado
- agregação via cosine similarity

Limitação conhecida: **c_v penaliza K alto** (Stevens et al. 2012). Por isso complementamos com inspeção qualitativa em múltiplos K (próxima célula).

In [ ]:
# Cell 8 — Grid search K (carrega resultado do run standalone se existir)
out_dir = f"../data/output/{CORPUS_ID}"
m_path = f"{out_dir}/lda_metrics.csv"

if os.path.exists(m_path):
    m = pd.read_csv(m_path)
    scores = ast.literal_eval(m.iloc[0]["k_grid_scores"])
    scores = {int(k): float(v) for k, v in scores.items()}
    print(f"Carregado grid pre-computado de {m_path}")
    print(f"  K range: {sorted(scores.keys())}")
    print(f"  passes: {m.iloc[0].get('grid_passes', '?')}")
else:
    K_RANGE = [3, 5, 7, 8, 10, 12, 15, 20, 25, 30]
    print(f"Rodando grid K em {K_RANGE} passes=20 (~13min)...")
    scores = grid_search_k(corpus_bow, dictionary, tokenized, K_RANGE, seed=seed, passes=20)
    print(f"Done.")

scores

In [ ]:
# Cell 9 — Visualizacao c_v vs K
fig, ax = plt.subplots(figsize=(11, 5))
ks = sorted(scores)
vs = [scores[k] for k in ks]

ax.plot(ks, vs, marker="o", linewidth=2.5, markersize=9, color="steelblue")
ax.fill_between(ks, vs, alpha=0.1, color="steelblue")

peak_k = max(scores, key=scores.get)
ax.axvline(peak_k, color="red", linestyle="--", alpha=0.6, label=f"peak c_v: K={peak_k} ({scores[peak_k]:.3f})")
ax.axvline(10, color="green", linestyle=":", alpha=0.7, linewidth=2, label="K=10 (top-10 ModApte canonical)")

# Annotate each K
for k, v in zip(ks, vs):
    delta = (v - scores[peak_k]) / scores[peak_k] * 100
    label = f"{v:.3f}" if k == peak_k else f"{v:.3f}\n({delta:+.0f}%)"
    ax.annotate(label, (k, v), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=8)

ax.set_xlabel("K (número de tópicos)"); ax.set_ylabel("c_v coherence")
ax.set_title(f"LDA grid search — Reuters-21578 ({len(docs):,} docs)")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

print(f"\nPeak: K={peak_k} (c_v={scores[peak_k]:.4f})")
if 10 in scores:
    print(f"K=10: c_v={scores[10]:.4f} (delta vs peak: {(scores[10]-scores[peak_k])/scores[peak_k]*100:+.1f}%)")

## 5. Inspeção qualitativa multi-K

Como `c_v` favorece K menor, precisamos olhar **os tópicos em si** para escolher K com critério metodológico (não só estatístico). Carregamos os tópicos de K=5, 7, 10, 15, 30 já treinados via `inspect_lda_reuters_multi_k.py`.

In [ ]:
# Cell 10 — Multi-K topics side by side (carregado de inspect_lda_reuters_multi_k.py)
multi_path = f"{out_dir}/lda_multi_k_inspect.csv"
if os.path.exists(multi_path):
    multi = pd.read_csv(multi_path)
    for k_val in sorted(multi["k"].unique()):
        sub = multi[multi["k"] == k_val].sort_values("topic_id")
        print(f"=== K={k_val} ===")
        for _, row in sub.iterrows():
            kws = row["keywords"].split(", ")[:8]
            print(f"  T{int(row['topic_id']):2d}: {', '.join(kws)}")
        print()
else:
    print(f"Rode antes: python ../inspect_lda_reuters_multi_k.py")

In [ ]:
# Categorias canônicas da Folha para validação
# (equivalente ao ModApte top-10 do Reuters)
if 'category' in df.columns:
    cat_dist = df['category'].value_counts()
    print('Distribuicao de categorias:')
    print(cat_dist.to_string())
else:
    print('Coluna category nao encontrada — pular validacao por categoria')


## 6. Decisão de K e treino final

**Trade-off:** `c_v` favorece K pequeno. Combine com inspeção qualitativa.
Ajuste `BEST_K` na célula abaixo se necessário.


In [ ]:
# Cell 12 — Treina LDA com K final
BEST_K = 10  # ajuste aqui: 5 (peak c_v) | 10 (recomendado, top-10 ModApte) | 15 | 30

print(f"Treinando LDA K={BEST_K}, passes=20 (~30s-1min)...")
t0 = time.time()
model = train_lda(corpus_bow, dictionary, k=BEST_K, seed=seed, passes=20)
print(f"  done em {time.time()-t0:.0f}s")

top_n = params["evaluation"]["top_n_keywords"]
topics_keywords = extract_topics_keywords(model, k=BEST_K, top_n=top_n)

print(f"\nTopics (top-{top_n}):")
for tid in sorted(topics_keywords):
    print(f"  T{tid:2d}: {topics_keywords[tid]}")

## 7. Visualização dos tópicos

Wordclouds + bar charts mostram a estrutura semântica de cada tópico. Escala logarítmica destaca palavras menos frequentes mas igualmente representativas.

In [ ]:
# Cell 13 — Wordclouds (1 por topico)
ncols = 3
nrows = (BEST_K + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten() if BEST_K > 1 else [axes]

for tid in sorted(topics_keywords):
    # gensim show_topic retorna (palavra, prob); usar prob como freq do wordcloud
    word_probs = dict(model.show_topic(tid, topn=30))
    wc = WordCloud(width=600, height=400, background_color="white",
                   colormap="viridis", max_words=30, prefer_horizontal=0.9).generate_from_frequencies(word_probs)
    axes[tid].imshow(wc, interpolation="bilinear")
    axes[tid].set_title(f"T{tid}", fontsize=12, fontweight="bold")
    axes[tid].axis("off")

# Esconde axes nao usados
for i in range(BEST_K, len(axes)):
    axes[i].axis("off")

plt.tight_layout(); plt.show()

In [ ]:
# Cell 14 — Bar chart top-10 keywords per topic
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.2 * nrows))
axes = axes.flatten() if BEST_K > 1 else [axes]

for tid in sorted(topics_keywords):
    word_probs = model.show_topic(tid, topn=10)
    words, probs = zip(*word_probs)
    axes[tid].barh(list(words)[::-1], list(probs)[::-1], color="steelblue")
    axes[tid].set_title(f"T{tid}", fontsize=11, fontweight="bold")
    axes[tid].set_xlabel("P(palavra | tópico)")
    axes[tid].tick_params(labelsize=9)

for i in range(BEST_K, len(axes)):
    axes[i].axis("off")

plt.tight_layout(); plt.show()

In [ ]:
import seaborn as sns
# Heatmap phi: top palavras x topicos
_top_n_phi = 25
_all_words = []
for tid in range(BEST_K):
    for w, _ in model.show_topic(tid, topn=_top_n_phi):
        if w not in _all_words:
            _all_words.append(w)
_all_words = _all_words[:min(50, len(_all_words))]

_phi = np.zeros((len(_all_words), BEST_K))
for tid in range(BEST_K):
    _td = dict(model.show_topic(tid, topn=200))
    for wi, w in enumerate(_all_words):
        _phi[wi, tid] = _td.get(w, 0.0)

import textwrap as _tw_lda
_col_labels = [f"T{t}:\n{_tw_lda.shorten(str(topics_names.get(t,"")), 18, placeholder="...")}" for t in range(BEST_K)]
_phi_df = pd.DataFrame(_phi, index=_all_words, columns=_col_labels)

fig, ax = plt.subplots(figsize=(max(10, BEST_K * 0.8), max(8, len(_all_words) * 0.32)))
sns.heatmap(_phi_df, cmap="YlOrRd", ax=ax, linewidths=0.15,
            cbar_kws={"label": "p(word|topic) phi"})
ax.set_title("Distribuicao phi — palavras x topicos (LDA)", fontsize=12, fontweight="bold")
ax.set_xlabel("Topico"); ax.set_ylabel("Palavra")
plt.xticks(fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_heatmap_phi.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_heatmap_phi.png'}")


In [ ]:
import seaborn as sns
# Similaridade cosseno entre topicos (vetores phi)
from sklearn.metrics.pairwise import cosine_similarity

_phi_topics = np.zeros((BEST_K, len(dictionary)))
for tid in range(BEST_K):
    for word_id, prob in model.get_topic_terms(tid, topn=len(dictionary)):
        _phi_topics[tid, word_id] = prob

_cos_sim = cosine_similarity(_phi_topics)
import textwrap as _tw_cos
_tlabels = [f"T{t}: {_tw_cos.shorten(str(topics_names.get(t,"")), 20, placeholder="...")}" for t in range(BEST_K)]

fig, ax = plt.subplots(figsize=(max(8, BEST_K * 0.65), max(7, BEST_K * 0.6)))
sns.heatmap(_cos_sim, xticklabels=_tlabels, yticklabels=_tlabels,
            cmap="RdYlGn_r", vmin=0, vmax=1,
            annot=(BEST_K <= 20), fmt=".2f", ax=ax,
            annot_kws={"fontsize": 7},
            cbar_kws={"label": "Cosine similarity (phi)"})
ax.set_title("Similaridade entre topicos (cosine phi) — LDA", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_heatmap_topicos_cosine.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_heatmap_topicos_cosine.png'}")


## 8. Rotulagem via LLM (gemma4:e4b)

`src.naming.name_all_topics` envia as keywords pra `gemma4:e4b` via Ollama com prompt few-shot. Em ~9s/tópico em CPU. Fallback: top-3 keywords se Ollama indisponível.

⚠ **Nota**: gemma4 traduz nomes para PT-BR mesmo em corpus EN. Para nomes em EN, ajustar prompt em `src/naming.py`.

In [ ]:
# Cell 15 — Naming
llm_cfg = params.get("llm", {})
print(f"Naming via {llm_cfg.get('model')} ({BEST_K} tópicos × ~9s = ~{BEST_K*9}s)...")

t0 = time.time()
try:
    names = name_all_topics(
        topics_keywords,
        model=llm_cfg.get("model", "gemma4:e4b"),
        base_url=llm_cfg.get("base_url", "http://localhost:11434"),
    )
    print(f"  done em {time.time()-t0:.0f}s")
except Exception as e:
    print(f"  LLM falhou ({e}) — fallback top-3")
    names = {tid: ", ".join(kws[:3]) for tid, kws in topics_keywords.items()}

print()
for tid in sorted(names):
    print(f"  T{tid:2d}: {names[tid]}")
    print(f"        keywords: {topics_keywords[tid][:8]}")

## 9. Métricas quantitativas

- **c_v** recomputada (validação cruzada do grid)
- **Exclusivity**: 1 - prop. de keywords compartilhadas entre tópicos
- **Topic Diversity**: razão de palavras únicas no consolidado dos top-K
- **Diversity entropy** das distribuições θ doc-tópico (quão espalhadas)

In [ ]:
# Cell 16 — Calcula metricas
print("Calculando métricas...")
dominant, full = compute_doc_distributions(model, corpus_bow, k=BEST_K)

metrics = {
    "model": "lda",
    "topic_type": "probabilistic",
    "granularity": "unit",
    "corpus": CORPUS_ID,
    "n_topics": BEST_K,
    "best_k_cv": float(scores.get(BEST_K, 0.0)),
    "exclusivity": float(compute_exclusivity(topics_keywords)),
    "diversity_entropy": float(compute_diversity(full)),
    "topic_diversity_dieng": float(compute_topic_diversity(topics_keywords)),
}
metrics["coherence_cv_recomputed"] = float(compute_coherence_cv(topics_keywords, tokenized, dictionary))

# Visualiza num radar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
metric_keys = ["best_k_cv", "coherence_cv_recomputed", "exclusivity",
               "topic_diversity_dieng", "diversity_entropy"]
metric_vals = [metrics[k] for k in metric_keys]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
axes[0].bar(metric_keys, metric_vals, color=colors)
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("Score (0-1)")
axes[0].set_title(f"Metricas LDA K={BEST_K}")
axes[0].tick_params(axis="x", rotation=20, labelsize=9)
for i, (k, v) in enumerate(zip(metric_keys, metric_vals)):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")

# Radar
angles = np.linspace(0, 2*np.pi, len(metric_keys), endpoint=False).tolist()
vals = metric_vals + [metric_vals[0]]
angles_p = angles + [angles[0]]
axes[1] = plt.subplot(122, projection="polar")
axes[1].plot(angles_p, vals, "o-", linewidth=2, color="steelblue")
axes[1].fill(angles_p, vals, alpha=0.25, color="steelblue")
axes[1].set_xticks(angles); axes[1].set_xticklabels(metric_keys, fontsize=8)
axes[1].set_ylim(0, 1)
axes[1].set_title("Visao radar")

plt.tight_layout(); plt.show()
print(json.dumps(metrics, indent=2))

In [ ]:
# Cell 17 — c_v por topico (nao so media)
from gensim.models import CoherenceModel
cm = CoherenceModel(
    topics=[topics_keywords[tid] for tid in sorted(topics_keywords)],
    texts=tokenized, dictionary=dictionary, coherence="c_v",
)
per_topic_cv = cm.get_coherence_per_topic()

fig, ax = plt.subplots(figsize=(11, 4))
tids = sorted(topics_keywords)
labels = [f"T{tid}\n{names[tid][:22]}" for tid in tids]
bars = ax.bar(tids, per_topic_cv, color=["#2ca02c" if c >= 0.5 else "#ff7f0e" if c >= 0.4 else "#d62728" for c in per_topic_cv])
ax.axhline(np.mean(per_topic_cv), color="black", linestyle="--", alpha=0.5, label=f"média={np.mean(per_topic_cv):.3f}")
ax.set_xticks(tids); ax.set_xticklabels(labels, fontsize=8, rotation=30, ha="right")
ax.set_ylabel("c_v coherence"); ax.set_title("Coerência c_v por tópico (verde >=0.5, laranja 0.4-0.5, vermelho <0.4)")
ax.legend()
for bar, c in zip(bars, per_topic_cv):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{c:.3f}",
            ha="center", fontsize=8)
plt.tight_layout(); plt.show()

In [13]:
stability_seeds = params["evaluation"]["stability_seeds"]
results_per_seed = {}

for s in stability_seeds:
    print(f"Executando seed={s}...")
    model_s = LdaMulticore(
        corpus=corpus_bow,
        id2word=dictionary,
        num_topics=BEST_K,
        random_state=s,
        passes=20,
        workers=2,
    )
    seed_keywords = {}
    for tid in range(BEST_K):
        words = model_s.show_topic(tid, topn=top_n)
        seed_keywords[tid] = [w for w, _ in words]
    results_per_seed[s] = seed_keywords

    seed_topics = []
    for bow in corpus_bow:
        dist = model_s.get_document_topics(bow, minimum_probability=0.0)
        seed_topics.append(max(dist, key=lambda x: x[1])[0])
    seed_df = pd.DataFrame({"doc_id": range(len(seed_topics)), "topic_id": seed_topics})
    seed_df.to_csv(OUTPUT_DIR / f"lda_topics_seed{s}.csv", index=False)

mean_j, std_j = compute_stability(results_per_seed)
print("\nEstabilidade LDA:")
print(f"  Jaccard médio: {mean_j:.4f}")
print(f"  Desvio padrão: {std_j:.4f}")


Executando seed=42...


Executando seed=123...


Executando seed=456...


Executando seed=789...


Executando seed=1024...



Estabilidade LDA:
  Jaccard médio: 0.3284
  Desvio padrão: 0.0331


## Estabilidade — Jaccard por tópico

Compara keywords do modelo principal (seed=42) com os 4 seeds alternativos. Jaccard < 0.5 indica tópico instável.

In [ ]:
import seaborn as sns
# Jaccard de estabilidade por topico (seeds vs referencia)
import textwrap as _tw_stab

_ref_kws = topics_keywords  # seed principal (SEED)
_other_seeds = [s for s in results_per_seed if s != SEED]
_jac_rows = []
for s in _other_seeds:
    _skws = results_per_seed[s]
    for tid in range(BEST_K):
        _r = set(_ref_kws.get(tid, []))
        _o = set(_skws.get(tid, []))
        _u = _r | _o
        j = len(_r & _o) / len(_u) if _u else 0.0
        _jac_rows.append({"seed": str(s), "topic_id": tid, "jaccard": j})

_jac_df = pd.DataFrame(_jac_rows)
_topic_order = (_jac_df.groupby("topic_id")["jaccard"].mean()
                .sort_values(ascending=True).index.tolist())
_jlabels = {t: f"T{t}: {_tw_stab.shorten(str(topics_names.get(t,"")), 30, placeholder="...")}" for t in range(BEST_K)}
_jac_df["topic_label"] = _jac_df["topic_id"].map(_jlabels)
_ordered = [_jlabels[t] for t in _topic_order]

fig, ax = plt.subplots(figsize=(9, max(5, 0.45 * BEST_K)))
sns.boxplot(data=_jac_df, x="jaccard", y="topic_label", order=_ordered,
            palette="RdYlGn", orient="h", ax=ax)
ax.axvline(x=0.5, color="red", linestyle="--", alpha=0.5, linewidth=1,
           label="limiar 0.5")
ax.set_xlabel("Jaccard (vs seed referencia)", fontsize=10)
ax.set_title(f"Estabilidade por topico (LDA, {len(_other_seeds)} seeds)", fontsize=12, fontweight="bold")
ax.set_xlim(0, 1)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_stability_boxplot.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_stability_boxplot.png'}")


## 10. Distribuição de documentos por tópico

Mostra balanceamento dos clusters. Distribuição muito desigual pode indicar:
- 1 tópico "lixo" capturando outliers
- K muito alto fragmentando o sinal
- Categorias do corpus naturalmente desbalanceadas (Reuters: `earn` é ~40% dos docs)

In [ ]:
# Cell 18 — Bar chart docs por topico
topic_doc_counts = pd.Series(dominant).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 4.5))
labels = [f"T{tid}\n{names[tid][:22]}" for tid in topic_doc_counts.index]
bars = ax.bar(topic_doc_counts.index, topic_doc_counts.values, color="steelblue")
ax.set_xticks(topic_doc_counts.index); ax.set_xticklabels(labels, fontsize=8, rotation=30, ha="right")
ax.set_ylabel("# documentos dominantes")
ax.set_title(f"Distribuição de docs por tópico (n={len(docs):,})")
for bar, c in zip(bars, topic_doc_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, c + max(topic_doc_counts)*0.01,
            f"{c:,}\n({c/len(docs)*100:.1f}%)", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

print(f"\nDocs por tópico — descritivas: {topic_doc_counts.describe().to_dict()}")

In [ ]:
# Cell 19 — Heatmap theta (doc x topico) — sample
sample_n = min(200, len(full))
sample_idx = np.random.RandomState(42).choice(len(full), sample_n, replace=False)
theta_sample = np.array([full[i] for i in sample_idx])

# Sort sample by dominant topic for readability
sample_dom = theta_sample.argmax(axis=1)
order = np.argsort(sample_dom)
theta_sorted = theta_sample[order]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(theta_sorted.T, cmap="YlOrRd", cbar_kws={"label": "P(tópico | doc)"},
            xticklabels=False, yticklabels=[f"T{i}" for i in range(BEST_K)], ax=ax)
ax.set_xlabel(f"Documentos (sample {sample_n}, ordenados por tópico dominante)")
ax.set_ylabel("Tópico"); ax.set_title("Distribuição θ (doc → tópico)")
plt.tight_layout(); plt.show()

## UMAP do espaço de tópicos (θ)

Reduz a matriz θ (N×K) a 2D com métrica de Hellinger. Gera PNG colorido por tópico dominante, por sentimento e HTML interativo.

In [ ]:
import plotly.express as px
from sklearn.manifold import TSNE

_theta_mat = np.array(topic_distributions)  # (n_docs, K)
_sent_vals_lda = df["sentiment"].fillna("neutral").values
_likes_lda = df["likes"].fillna(0).values.astype(int)
_texts_lda = [t[:100] + "..." if len(t) > 100 else t for t in docs]

print("Calculando t-SNE sobre matriz theta...")
_tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=500)
_theta_2d = _tsne.fit_transform(_theta_mat)

_SENT_C_LDA = {"positive": "#4caf50", "negative": "#f44336", "neutral": "#9e9e9e"}
_cmap_lda = plt.get_cmap("tab20", max(best_k, 1))

# PNG por topico dominante
fig, ax = plt.subplots(figsize=(11, 8))
for tid in range(best_k):
    _mm = np.array(dominant_topics) == tid
    if _mm.any():
        import textwrap as _tw_u
        _nm = _tw_u.shorten(str(topics_names.get(tid, "")), 20, placeholder="...")
        ax.scatter(_theta_2d[_mm, 0], _theta_2d[_mm, 1],
                   c=[_cmap_lda(tid)], s=14, alpha=0.72, linewidths=0,
                   label=f"T{tid}: {_nm}")
ax.set_title("t-SNE do espaco theta — topico dominante (LDA)", fontsize=13, fontweight="bold")
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
ax.legend(loc="best", fontsize=7, framealpha=0.85, ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_tsne_theta_topico.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_tsne_theta_topico.png'}")

# PNG por sentimento
fig, ax = plt.subplots(figsize=(11, 8))
for sent, color in _SENT_C_LDA.items():
    _ms = _sent_vals_lda == sent
    if _ms.any():
        ax.scatter(_theta_2d[_ms, 0], _theta_2d[_ms, 1],
                   c=color, s=14, alpha=0.72, linewidths=0,
                   label=f"{sent.capitalize()} (n={_ms.sum()})")
ax.set_title("t-SNE do espaco theta — sentimento (LDA)", fontsize=13, fontweight="bold")
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
ax.legend(loc="best", fontsize=10, framealpha=0.85)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_tsne_theta_sentiment.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_tsne_theta_sentiment.png'}")

# HTML interativo por sentimento
import textwrap as _tw_u2
_tsne_df_lda = pd.DataFrame({
    "x": _theta_2d[:, 0], "y": _theta_2d[:, 1],
    "Sentimento": _sent_vals_lda, "Likes": _likes_lda, "Texto": _texts_lda,
    "Topico": [f"T{t}: {_tw_u2.shorten(str(topics_names.get(t,"")), 25, placeholder="...")}" for t in dominant_topics],
})
fig_u = px.scatter(
    _tsne_df_lda, x="x", y="y", color="Sentimento",
    color_discrete_map=_SENT_C_LDA,
    hover_data={"x": False, "y": False, "Topico": True, "Likes": True, "Texto": True},
    title="t-SNE espaco theta — sentimento (interativo, LDA)",
    labels={"x": "t-SNE dim 1", "y": "t-SNE dim 2"}, opacity=0.72,
)
fig_u.update_traces(marker_size=5)
_out_u = OUTPUT_DIR / "lda_tsne_theta_interactive.html"
fig_u.write_html(_out_u)
fig_u.show()
print(f"Salvo: {_out_u}")


## 11. Cross-tab tópico × categoria Folha

Folha tem categorias editoriais (ex: `politica`, `economia`, `esporte`, `mundo`).
Concentração de um tópico em uma categoria valida coerência semântica.


In [ ]:
# Heatmap tópico × categoria — folha
if 'category' in df.columns:
    df_topic = df[['category']].copy()
    df_topic['topic_id'] = dominant
    df_topic = df_topic[df_topic['topic_id'] != -1]
    pivot = df_topic.groupby(['category', 'topic_id']).size().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns)*0.8), max(4, len(pivot)*0.5)))
    sns.heatmap(pivot_pct.round(1), annot=True, fmt='.1f', cmap='Blues',
                linewidths=0.5, ax=ax)
    ax.set_title('% documentos por categoria × tópico — folha')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'lda_topic_category.png', dpi=150)
    plt.show()
else:
    print('Coluna category nao encontrada')


## 12. pyLDAvis — visualização interativa

Visualização canônica do LDA (Sievert & Shirley 2014). Mostra:
- **Esquerda**: tópicos como círculos no plano 2D (PCA do espaço inter-tópico). Tamanho = prevalência. Distância = dissimilaridade.
- **Direita**: bar chart das top-30 palavras do tópico selecionado, com slider λ controlando "exclusivity vs frequency" (Bischof & Airoldi 2016 — base do FREX score).

In [ ]:
# Cell 21 — pyLDAvis (interativo, salva HTML)
import pyLDAvis
import pyLDAvis.gensim_models as gensim_pyLDAvis

pyLDAvis.enable_notebook()

print("Preparing pyLDAvis (~30s)...")
vis_data = gensim_pyLDAvis.prepare(model, corpus_bow, dictionary, sort_topics=False)
out_html = f"{out_dir}/lda_pyldavis_K{BEST_K}.html"
pyLDAvis.save_html(vis_data, out_html)
print(f"saved: {out_html}")
vis_data

## 13. Top documentos por tópico

Mostra os 3 docs com maior probabilidade em cada tópico — verificação qualitativa direta da semântica.

In [ ]:
# Cell 22 — Top 3 docs por topico
TOP_N_DOCS = 3
TRUNC = 200

theta_arr = np.array(full)
print("Top docs por topico (truncados em 200 chars):\n")
for tid in sorted(topics_keywords):
    top_idx = np.argsort(theta_arr[:, tid])[::-1][:TOP_N_DOCS]
    print(f"=== T{tid} [{names[tid]}] ===")
    print(f"  keywords: {topics_keywords[tid][:8]}\n")
    for rank, idx in enumerate(top_idx, 1):
        prob = theta_arr[idx, tid]
        cat = df.iloc[idx]["category"]
        text = df.iloc[idx]["message"][:TRUNC].replace("\n", " ")
        print(f"  #{rank} (P={prob:.3f}, cat={cat}): {text}...")
        print()

## 14. Export + relatório qualitativo

Gera os CSVs canônicos (`lda_results.csv`, `lda_topics_for_eval.csv`, `lda_metrics.csv`) e atualiza o template em `docs/baselines/lda_reuters_baseline.md` com a leitura preenchida.

In [ ]:
# Cell 23 — Export final
os.makedirs(out_dir, exist_ok=True)

export_results(
    topics=dominant, probs=full, names=names, texts=docs,
    output_path=f"{out_dir}/lda_results.csv",
    topic_type="probabilistic", granularity="unit", post_ids=post_ids,
)
export_topics_for_eval(
    topics_keywords=topics_keywords, topics_names=names,
    model_name="lda", output_path=f"{out_dir}/lda_topics_for_eval.csv",
)
metrics_full = {**metrics, "k_grid_scores": json.dumps(scores), "grid_passes": 20}
pd.DataFrame([metrics_full]).to_csv(f"{out_dir}/lda_metrics.csv", index=False)

print(f"CSVs em {out_dir}/:")
for f in ["lda_results.csv", "lda_topics_for_eval.csv", "lda_metrics.csv"]:
    sz = os.path.getsize(f"{out_dir}/{f}")
    print(f"  {f} ({sz/1024:.0f} KB)")
print(f"  lda_pyldavis_K{BEST_K}.html")

In [ ]:
# Cell 24 — Relatorio qualitativo (template)
md = qualitative_report(topics_keywords, names)
print(md)

## Próximos passos
- Comparar com BERTopic (`01_bertopic_folha.ipynb`) — triangulação LDA × BERTopic
- Analisar evolução temporal de tópicos (coluna `date` disponível)
